# C — Tutor answers (GPU)

Generates the answers the verifier will be judged on. Each of the 170 curriculum
questions is put to four models spanning a capability range — Qwen2.5 at 0.5B, 1.5B, 3B
and 7B — **closed-book**, with no access to the textbook.

Closed-book is the point: an ungrounded model answering a Class 9–10 Biology question is
exactly the situation a curriculum verifier is meant to catch.

**Why four sizes rather than one good one.** A single weak model would show a large gain
for graph grounding and invite the objection that the baseline was chosen to flatter the
method. A ladder lets you show the gain holds across capability while shrinking as models
improve, which is the more defensible result.

**Settings:** GPU `NvidiaTeslaT4`, Internet on, dataset `bangla-biology-kg`,
models `qwen-lm/qwen2.5` at `0.5b-instruct`, `1.5b-instruct`, `3b-instruct`, `7b-instruct`.

Answers checkpoint per (model, question), so an interrupted session resumes.

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate

import importlib.util
assert importlib.util.find_spec("bitsandbytes"), "bitsandbytes missing — restart the session"
print("bitsandbytes ready")

In [ ]:
import json, glob, os, gc, time
from pathlib import Path
import pandas as pd

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- config -------------------------------------------------------------
MODELS   = ["0.5b-instruct", "1.5b-instruct", "3b-instruct", "7b-instruct"]
LIMIT    = None     # questions per model; None = all 170
MAX_NEW  = 256      # a doubt-solving answer is short, not an essay
BATCH    = 8
# -------------------------------------------------------------------------

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
ANSWERS_JL = WORK / "tutor_answers.jsonl"

hits = (glob.glob("/kaggle/input/**/biology_eval_questions.csv", recursive=True)
        or glob.glob("eval/biology_eval_questions.csv")
        or glob.glob("../eval/biology_eval_questions.csv"))
if not hits:
    raise SystemExit("biology_eval_questions.csv not found — attach the bangla-biology-kg dataset.")

Q = pd.read_csv(hits[0])
if LIMIT:
    Q = Q.head(LIMIT)
print(f"{len(Q)} questions, {Q.chapter_no.nunique()} chapters")
print(f"{len(MODELS)} models -> {len(Q) * len(MODELS)} generations")

## Prompt

The model is cast as a Class 9–10 Biology tutor answering a student's doubt, and asked for
a short Bangla answer. Nothing instructs it to hedge or refuse — a tutor that says "I am
not sure" produces no claim to verify, and the experiment is about what an ungrounded
tutor asserts.

In [ ]:
SYSTEM = ("তুমি বাংলাদেশের নবম-দশম শ্রেণির জীববিজ্ঞান বিষয়ের একজন শিক্ষক। "
          "শিক্ষার্থীদের প্রশ্নের উত্তর সহজ বাংলায় দাও।")

PROMPT = """নিচের প্রশ্নটির উত্তর দাও।

নিয়ম:
- উত্তর বাংলায় লিখবে।
- সংক্ষেপে, ৩-৪ বাক্যের মধ্যে উত্তর দেবে।
- পাঠ্যবইয়ের তথ্য অনুযায়ী উত্তর দেবে।

প্রশ্ন: {question}"""


def build(tok, q):
    return tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user", "content": PROMPT.format(question=q)}],
        tokenize=False, add_generation_prompt=True)


print(PROMPT.format(question=Q.question_text.iloc[0]))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


def load(size):
    """Kaggle mounts models at /kaggle/input/models/<owner>/<model>/<framework>/<variation>.
    Fail loudly on a miss rather than silently loading whichever model turns up first."""
    cfgs = glob.glob(f"/kaggle/input/**/{size}/**/config.json", recursive=True)
    if not cfgs:
        raise SystemExit(f"{size} not attached")
    path = str(Path(cfgs[0]).parent)
    assert f"/{size}/" in path + "/", f"resolved wrong model: {path}"

    tok = AutoTokenizer.from_pretrained(path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        path,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True),
        device_map="auto",
    ).eval()
    return tok, model, path


def unload(model):
    """Four models in one session: without this the second load OOMs."""
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
done = set()
if ANSWERS_JL.exists():
    with open(ANSWERS_JL, encoding="utf-8") as f:
        done = {(r["model"], r["qid"]) for r in map(json.loads, filter(str.strip, f))}
    print(f"resuming — {len(done)} answers already generated")


@torch.inference_mode()
def generate(tok, model, prompts):
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
            for o in out]


t_all = time.time()
with open(ANSWERS_JL, "a", encoding="utf-8") as sink:
    for size in MODELS:
        todo = Q[[(size, q) not in done for q in Q.qid]].reset_index(drop=True)
        if not len(todo):
            print(f"{size}: already complete")
            continue

        tok, model, path = load(size)
        print(f"\n{size}: {len(todo)} to generate  [{path}]")
        t0 = time.time()

        for i in range(0, len(todo), BATCH):
            part = todo.iloc[i:i + BATCH]
            prompts = [build(tok, q) for q in part.question_text]
            try:
                answers = generate(tok, model, prompts)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                print("\n  OOM — retrying one at a time")
                answers = [generate(tok, model, [p])[0] for p in prompts]
            for row, ans in zip(part.itertuples(), answers):
                sink.write(json.dumps({
                    "model": size, "qid": row.qid, "chapter_no": int(row.chapter_no),
                    "question": row.question_text, "answer": ans,
                }, ensure_ascii=False) + "\n")
            sink.flush()
            n = min(i + BATCH, len(todo))
            el = time.time() - t0
            print(f"  {n}/{len(todo)}  {el:.0f}s  ({el/max(n,1):.1f}s/q)", end="\r")

        print(f"\n  {size} done in {time.time()-t0:.0f}s")
        unload(model)

print(f"\nall models done in {time.time()-t_all:.0f}s")

## Collect

In [ ]:
rows = [json.loads(l) for l in open(ANSWERS_JL, encoding="utf-8") if l.strip()]
A = pd.DataFrame(rows).drop_duplicates(subset=["model", "qid"])
A["n_chars"] = A.answer.str.len()
A.to_csv(WORK / "tutor_answers.csv", index=False)

print(f"{len(A)} answers  ({A.model.nunique()} models x {A.qid.nunique()} questions)")
print(A.groupby("model").agg(answers=("qid", "count"),
                             median_chars=("n_chars", "median"),
                             empty=("n_chars", lambda s: int((s < 5).sum()))).to_string())

# A model that answers in the wrong script is a finding, not a bug to hide.
import re
bn = re.compile(r"[ঀ-৿]")
A["bangla_ratio"] = A.answer.map(
    lambda s: len(bn.findall(s)) / max(len(re.sub(r"\s", "", s)), 1))
print("\nmedian share of Bangla characters per answer:")
print(A.groupby("model").bangla_ratio.median().round(2).to_string())

In [ ]:
for size in MODELS:
    sub = A[A.model == size]
    if not len(sub):
        continue
    r = sub.iloc[0]
    print(f"\n=== {size} ===")
    print(f"Q: {r.question[:110]}")
    print(f"A: {str(r.answer)[:340]}")